In [0]:
bronze = "/Volumes/workspace/default/de_lab/medallion/bronze_sales"
silver = "/Volumes/workspace/default/de_lab/medallion/silver_sales"
gold   = "/Volumes/workspace/default/de_lab/medallion/gold_revenue_by_category"

In [0]:
fact_rows = [
    (1, 1, 2, 50.00, "COMPLETED"),
    (2, 2, 1, 40.00, "COMPLETED"),
    (3, 4, 3, 36.00, "COMPLETED"),
    (4, 3, 1, 45.00, "COMPLETED"),
    (5, 1, 1, 25.00, "COMPLETED"),
    (6, 2, 2, 80.00, "PENDING"),
    (7, 5, 4, 72.00, "CANCELLED"),
    (8, 1, 1, 25.00, "COMPLETED"),
    (9, 5, 2, 36.00, "COMPLETED"),
    (10, 2, 1, 40.00, "COMPLETED"),
]
fact_cols = ["sales_key", "product_key", "quantity", "line_amount", "status"]

In [0]:
product_rows = [
    (1, "Apparel"),
    (2, "Electronics"),
    (3, "Home"),
    (4, "Electronics"),
    (5, "Home"),
]
product_cols = ["product_key", "category"]

In [0]:
from pyspark.sql import functions as F

In [0]:
product_df = spark.createDataFrame(product_rows, product_cols)
fact_df = spark.createDataFrame(fact_rows, fact_cols)

In [0]:
fact_df.write.format("delta").mode("overwrite").save(bronze) ##BRONZE
spark.read.format("delta").load(bronze).show()

In [0]:
bronze_df = spark.read.format("delta").load(bronze) ##SILVER
completed_df = bronze_df.filter(F.col("status") == "COMPLETED")

joined_x = completed_df.join(product_df, on="product_key", how="inner")

silver_df = joined_x.select("sales_key", "product_key", "quantity", "line_amount", "status", "category")
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver)

spark.read.format("delta").load(silver).show()

In [0]:
silver_df = spark.read.format("delta").load(silver)

gold_df = (silver_df.groupBy("category")
           .agg(F.sum("line_amount").alias("revenue")))

gold_df.write.format("delta").mode("overwrite").save(gold)

spark.read.format("delta").load(gold).show()